In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import expon
from bayesian_dro.Bayesian_DRO_continuous import data_generation
from mis_dro.dataset import data_generation_outliers, data_generation_gamma, contaminated_normal
from bayesian_dro.Bayesian_DRO_continuous import EPSILON_SET
from mis_dro.models import univariate_GaussianModel
from mis_dro.npl import Npl

In [ ]:
experiment_dir = "/dcs/pg23/u1604520/misdro/results_kdro/"
experiment_dir_bdro = "/dcs/pg23/u1604520/misdro/results_kdro/"
# results_df = pd.read_csv(experiment_dir / "results.csv", index_col="uuid")
#results_other_dgps_df = pd.read_csv(experiment_dir / "results_patrick.csv", index_col="uuid")
#results_df = pd.concat([results_other_dgps_df, results_gamma_df])
dfs = []
# for dir in [experiment_dir, experiment_dir_bdro]:
for alg in ['mmd', 'bdro', 'kdro']: 
    for epsilon in EPSILON_SET:
        # if dir == experiment_dir:
        # if alg == 'kdro':
        if alg == 'kdro':
            dfs_k = []
            for i in range(5):
                if i == 0:
                    file_path = experiment_dir + f"{alg}_cont_exp_N900_ncert_200_cont_0.1_{epsilon}_fourier.csv"
                else:
                    file_path = experiment_dir + f"{alg}_cont_exp_N900_ncert_200_cont_0.1_{epsilon}_fourier_{i}.csv"
                df_ = pd.read_csv(file_path)
                dfs_k.append(df_)
            df = pd.concat(dfs_k, ignore_index=True)
            # file_path = experiment_dir + f"{alg}_cont_exp_N900_ncert_200_cont_0.1_{epsilon}_fourier.csv"
            # df = pd.read_csv(file_path)
        else:
            file_path = experiment_dir + f"{alg}_cont_exp_N900_ncert_200_cont_0.1_{epsilon}.csv"
            df = pd.read_csv(file_path)
            #f"kdro_cont_normal_N400_{epsilon}.csv"   #test_results_eps_{epsilon}.csv
        # else:
        # file_path = dir + f"bdro_exp_N100_ncert_20_{epsilon}.csv"
            # file_path = experiment_dir_bdro + f"{alg}_exp_N100_ncert_100_{epsilon}.csv"
        df['epsilon'] = epsilon
        # if dir == experiment_dir:
        # if alg == 'kdro':
        #     df['algorithm'] = 'mmd-dro'
        # else:
        #     df['algorithm'] = 'bdro'
        df['algorithm'] = alg
        dfs.append(df)
results_df = pd.concat(dfs, ignore_index=True)
# check the number of train/test observations is the same:
# assert len(results_df["num_observations"].unique()) == 1
# assert len(results_df["num_test_observations"].unique()) == 1
# num_observations = results_df["num_observations"]
# num_test_observations = results_df["num_test_observations"]
results_df

In [ ]:
results_df = results_df[results_df['epsilon'] < 0.2]

In [ ]:
gb = results_df.groupby(by=["algorithm", "epsilon"])
df = gb.agg({
    "mean_cost": ["mean", "var"],
    "var_cost": ["mean"],
    "posterior_time": ["mean"],
    "solve_time": ["mean"],
    "solution": ["mean", "var"]
})
df

In [ ]:
def plot_dgp(df, dgp: str, axis, Nsamples):
    # get a dataframe for each posterior
    bayes_df = df.loc["bdro", :]
    kdro_df = df.loc['kdro', :]
    mmd_df = df.loc["mmd", :]

    # # mean of variances + variance of means
    bayes_bootstrap_var = bayes_df["var_cost"]["mean"] + bayes_df["mean_cost"]["var"]
    kdro_bootstrap_var = kdro_df["var_cost"]["mean"] + kdro_df["mean_cost"]["var"]
    mmd_bootstrap_var = mmd_df["var_cost"]["mean"] + mmd_df["mean_cost"]["var"]

    # plot mean-variance trade-off
    axis.plot(bayes_bootstrap_var, bayes_df["mean_cost"]["mean"], marker='o', linestyle="solid", label="BDRO")
    axis.plot(kdro_bootstrap_var, kdro_df["mean_cost"]["mean"], marker="x", linestyle="dashed", label="BAS-DRO (MMD-NPL)")
    axis.plot(mmd_bootstrap_var, mmd_df["mean_cost"]["mean"], marker="v", linestyle="dotted", label="MMD-DRO")

    # set title and label epsilon
    axis.set_title(f"DGP is {dgp}, {Nsamples} samples")
    axis.legend()
    axis.set_xlabel("out-of-sample variance")
    axis.set_ylabel("out-of-sample mean")
    for i, epsilon in enumerate(mmd_df.index.get_level_values("epsilon")):
        if i % 4 == 0:
            axis.text(bayes_bootstrap_var[epsilon], bayes_df["mean_cost"]["mean"][epsilon], epsilon, ha='left', va='bottom')
        if (i+2) % 4 == 0:
            axis.text(mmd_bootstrap_var[epsilon], mmd_df["mean_cost"]["mean"][epsilon], epsilon, ha='right', va='bottom')

# plot mean-variance trade-off for each DGP
# dgp_list = list(df.index.get_level_values("dgp").unique())
# nrows = int(np.ceil(len(dgp_list)/2.0))
# fig, axes = plt.subplots(nrows=nrows, ncols=2, sharex=False, sharey=False, figsize=(10,10))
# for i, dgp in enumerate(dgp_list):
#     row = int(np.floor(i / 2.0))
#     col = i % 2
#     plot_dgp(dgp, axes[row][col])

In [ ]:
fig, axis = plt.subplots(nrows=1, ncols=1, sharex=False, sharey=False, figsize=(5,5))
plot_dgp(df, 'contaminated exponential', axis, 900)

In [ ]:
N = 20
contamination_level = 0.1
delta = 0.2
opt_epsilon = 4*contamination_level + 2*(np.sqrt(1/N) + np.sqrt((2*(N-1))/(N*(N+1))) + np.sqrt(2*N*np.log(1/delta))/N)
print(opt_epsilon)

In [ ]:
experiment_dir = "/dcs/pg23/u1604520/misdro/results_kdro/n100/"
experiment_dir_bdro = "/dcs/pg23/u1604520/misdro/results_kdro/n100/"
fig, axes = plt.subplots(nrows=1, ncols=1, sharex=False, sharey=False, figsize=(10,5))
N_values = [100] #, 400]
for i, N in enumerate(N_values):
    dfs = []
    for alg in ['bdro', 'kdro', 'mmd-dro']: #, 
        for epsilon in EPSILON_SET:
            if alg == 'bdro':
                file_path = experiment_dir_bdro + f"bdro_cont_exp_N100_ncert_200_cont_0.2_{epsilon}.csv" #f"kdro_cont_normal_N400_{epsilon}.csv"   #test_results_eps_{epsilon}.csv
            elif alg == 'kdro':
                file_path = experiment_dir + f"kdro_cont_exp_N100_ncert_200_cont_0.2_{epsilon}.csv"   # same exp just wrong file name
            else:
                file_path = experiment_dir + f"mmd_cont_exp_N{N}_ncert_200_cont_0.2_{epsilon}.csv" 
        
            df = pd.read_csv(file_path)
            df['epsilon'] = epsilon
            
            df['algorithm'] = alg
    
            dfs.append(df)
            
    results_df = pd.concat(dfs, ignore_index=True)
    results_df = results_df[results_df['epsilon'] < 0.2]
    gb = results_df.groupby(by=["algorithm", "epsilon"])
    df = gb.agg({
        "mean_cost": ["mean", "var"],
        "var_cost": ["mean"],
        "posterior_time": ["mean"],
        "solve_time": ["mean"],
        "solution": ["mean", "var"]
    })
    plot_dgp(df, 'contaminated_exp', axes, N)
plt.show()

In [ ]:
from scipy.stats import truncnorm
b = 8
h = 3
L = 100

DGP_MEAN_TRUNCATED_NORMAL = 10
DGP_STD_TRUNCATED_NORMAL = 10

NUM_OBSERVATIONS = 20
sol_true = truncnorm.ppf(
        (b - 0) / (h + b), a=-DGP_MEAN_TRUNCATED_NORMAL / DGP_STD_TRUNCATED_NORMAL, b=np.inf, loc=DGP_MEAN_TRUNCATED_NORMAL, scale=DGP_STD_TRUNCATED_NORMAL
    )

In [ ]:
sol_true

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, sharex=False, sharey=False, figsize=(5,5))
bayes_df = df.loc["bdro", :]
mmd_df = df.loc["mmd-dro", :]

bayes_bootstrap_var = bayes_df["solution"]["mean"] + bayes_df["solution"]["var"]
mmd_bootstrap_var = mmd_df["solution"]["mean"] + mmd_df["solution"]["var"]

# plot mean-variance trade-off
ax.plot(bayes_bootstrap_var, bayes_df["solution"]["mean"], marker='o', linestyle="solid", label="Bayesian")
    # axis.plot(wll_bootstrap_var, wll_df["mean_cost"]["mean"], marker="x", linestyle="dashed", label="WLL-NPL")
ax.plot(mmd_bootstrap_var, mmd_df["solution"]["mean"], marker="v", linestyle="dotted", label="MMD-NPL")

# set title and label epsilon
ax.set_title("DGP is contaminated exponential")
ax.legend()
ax.set_xlabel("solution variance")
ax.set_ylabel("solution mean")
for i, epsilon in enumerate(mmd_df.index.get_level_values("epsilon")):
    if i % 4 == 0:
        ax.text(bayes_bootstrap_var[epsilon], bayes_df["solution"]["mean"][epsilon], epsilon, ha='left', va='bottom')
    if (i+2) % 4 == 0:
        ax.text(mmd_bootstrap_var[epsilon], mmd_df["solution"]["mean"][epsilon], epsilon, ha='right', va='bottom')

Number of certifying points 

In [ ]:
n_cert_values = [5, 20, 100]
experiment_dir = "/dcs/pg23/u1604520/misdro/results_kdro/ncert/"
dfs = []
for ncert in n_cert_values:
    for epsilon in EPSILON_SET:
        file_path = experiment_dir + f"kdro_cont_exp_N25_ncert_{ncert}_{epsilon}.csv" 
        df = pd.read_csv(file_path)
        df['epsilon'] = epsilon
        df['ncert'] = ncert
        # df['algorithm'] = 'mmd-dro'
        dfs.append(df)
results_df = pd.concat(dfs, ignore_index=True)
gb = results_df.groupby(by=["epsilon", 'ncert'])
df = gb.agg({
    "mean_cost": ["mean", "var"],
    "var_cost": ["mean"],
    "posterior_time": ["mean"],
    "solve_time": ["mean"],
    "solution": ["mean", "var"]
})
df

In [ ]:
fig, axis = plt.subplots(nrows=1, ncols=1, sharex=False, sharey=False, figsize=(5,5))
for ncert in n_cert_values:
   dfn = df.xs(ncert, level='ncert')
   var = dfn["var_cost"]["mean"] + dfn["mean_cost"]["var"]
   axis.plot(var, dfn["mean_cost"]["mean"], marker="v", linestyle="dotted", label=f"n_cert_{ncert}")
axis.legend()
axis.set_xlabel("out-of-sample variance")
axis.set_ylabel("out-of-sample mean")

plt.show()

Tolerance level

In [ ]:
def calculate_opt_tol(num_observations, kernel_bound, concentration_param, delta, inf_mmd):
    N = num_observations
    M = kernel_bound
    a = concentration_param 
    first_term = np.sqrt(M/N) 
    second_term = np.sqrt((2*M*(N-1)+ a*(a+1))/((a+N)*(a+N+1)))
    third_term = np.sqrt((M*a*(a+1))/((a+N)*(a+N+1)))
    fourth_term = np.sqrt(2*N*M*np.log(1/delta))/(a+N)
    return 2*(first_term+second_term+third_term+fourth_term) + inf_mmd

In [ ]:
print(calculate_opt_tol(20,1,0,0.2,0))

### New

In [ ]:
experiment_dir = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/n80_outl_mvn")  #N20_contam0.1_exp_gauss_known_mean
experiment_dir_wellsp = Path("/dcs/pg23/u1604520/misdro/results_kdro/slurm_batches/N20_cont_mvn_known_cov") #N20_univ_normal_known_var
assert experiment_dir.exists()
results_df = pd.read_csv(experiment_dir / "results.csv", index_col=["uuid", "replication"])
results_df_2 = pd.read_csv(experiment_dir_wellsp / "results.csv", index_col=["uuid", "replication"])

num_total_samples = []
# for _, row in results_df.iterrows():
#     if row["algorithm"] == "kl_bdro":
#         num_total_samples.append(row["num_likelihood_samples"] ** 2)
#     elif row["algorithm"] == "kl_dro_bas":
#         num_total_samples.append(row["num_likelihood_samples"])
# results_df["num_total_samples"] = num_total_samples

# NOTE filter by DGP
results_df_005 = results_df.loc[results_df["contamination"] == 0.05]   #contaminated_exp contaminated_normal
results_df_wellsp = results_df_2.loc[results_df_2["dgp"] == "multivariate_normal"] 
results_df_01 = results_df.loc[results_df["contamination"] == 0.10] 

# results_df_02 = results_df_02[results_df_02['epsilon'] < 0.3]
# results_df

In [ ]:
pd.options.display.max_columns = None
results_df


In [ ]:
gb = results_df_005.groupby(by=["algorithm", "epsilon", "inference", "contamination"])
agg_df_005 = gb.agg({
    "mean_cost": ["mean", "var"],
    "var_cost": ["mean"],
    # "posterior_time": ["mean"],
    # "setup_time": ["mean"],
    # "solve_time": ["mean"],
    # "solution": ["mean", "var"],
})

gb = results_df_wellsp.groupby(by=["algorithm", "epsilon", "inference"])
agg_df_wellsp = gb.agg({
    "mean_cost": ["mean", "var"],
    "var_cost": ["mean"],
    # "posterior_time": ["mean"],
    # "setup_time": ["mean"],
    # "solve_time": ["mean"],
    # "solution": ["mean", "var"],
})

gb = results_df_01.groupby(by=["algorithm", "epsilon", "inference"])
agg_df_01 = gb.agg({
    "mean_cost": ["mean", "var"],
    "var_cost": ["mean"],
    # "posterior_time": ["mean"],
    # "setup_time": ["mean"],
    # "solve_time": ["mean"],
    # "solution": ["mean", "var"],
})

agg_df_01

In [ ]:
def plot_dgp(df, dgp: str, axis):
    # get a dataframe for each posterior
    bayes_df = df.loc["kl_bdro", :]
    bas_kl_df = df.loc['kl_dro_bas', :]
    bas_df = df.loc['dro_bas_mmd', :]
    empirical_df = df.loc["empirical_mmd", :]
    bdro_npl_df = df.loc["npl_bdro", :]

    # # mean of variances + variance of means
    bdro_bootstrap_var = bayes_df["var_cost"]["mean"] + bayes_df["mean_cost"]["var"]
    bas_bootstrap_var = bas_kl_df["var_cost"]["mean"] + bas_kl_df["mean_cost"]["var"]
    robas_bootstrap_var = bas_df["var_cost"]["mean"] + bas_df["mean_cost"]["var"]
    empirical_bootstrap_var = empirical_df["var_cost"]["mean"] + empirical_df["mean_cost"]["var"]
    bdro_npl_bootstrap_var =bdro_npl_df["var_cost"]["mean"] + bdro_npl_df["mean_cost"]["var"]

    # plot mean-variance trade-off
    # axis.plot(bdro_bootstrap_var, bayes_df["mean_cost"]["mean"], marker='o', linestyle="solid", label="BDRO")
    axis.plot(bas_bootstrap_var, bas_kl_df["mean_cost"]["mean"], marker='s', linestyle="solid", label="DRO-BAS")
    axis.plot(robas_bootstrap_var, bas_df["mean_cost"]["mean"], marker="x", linestyle="dashed", label="DRO-RoBAS")
    axis.plot(empirical_bootstrap_var, empirical_df["mean_cost"]["mean"], marker="+", linestyle="dotted", label="Empirical-MMD-DRO")
    axis.plot(bdro_npl_bootstrap_var, bdro_npl_df["mean_cost"]["mean"], marker="v", linestyle="dotted", label="BDRO-NPL")

    # set title and label epsilon
    axis.set_title(f"DGP is {dgp}")
    axis.legend()
    axis.set_xlabel("out-of-sample variance")
    axis.set_ylabel("out-of-sample mean")
    # for i, epsilon in enumerate(empirical_df.index.get_level_values("epsilon")):
    #     if i % 4 == 0:
    #         axis.text(bas_bootstrap_var[epsilon], bas_df["mean_cost"]["mean"][epsilon], epsilon, ha='left', va='bottom')
    #     if (i+2) % 4 == 0:
    #         axis.text(empirical_bootstrap_var[epsilon], empirical_df["mean_cost"]["mean"][epsilon], epsilon, ha='right', va='bottom')

In [ ]:
fig, axis = plt.subplots(nrows=1, ncols=3, sharex=False, sharey=False, figsize=(13,5))
plot_dgp(agg_df_wellsp, 'Well-specified', axis[0], 400)
plot_dgp(agg_df_005, '5% outliers', axis[1], 400)
plot_dgp(agg_df_01, '10% outliers', axis[2], 400)


In [ ]:
from mis_dro.models import *
from mis_dro.npl import Npl, sample_npl
from mis_dro.dataset import sample_dgp
from mis_dro.npl import *
contamination = 0.0
num_observations = 20
data = sample_dgp("cont_multivariate_normal", 20, 0.2, 5)
# model = multivariate_GaussianModel(num_observations, 5)
p = 5
dim =5
inference = "npl_mmd"
likelihood = "multivariate_normal"
num_posterior_samples = 20

theta_sample = sample_npl(
        data,
        inference,
        likelihood,
        num_posterior_samples,
        seed=0,
        lengthscale=-1,
        dim=dim,
        p=5
        )



In [ ]:
from mis_dro.dataset import *
data = sample_dgp('multivariate_normal', 50, 0.0, 5)
data.shape

In [ ]:
from mis_dro.dataset import *
from mis_dro.npl import *
replication = 0
generator = np.random.default_rng(seed=replication)
# NOTE if contamination is specified, then only contaminate the training samples (not test samples)
dim = 5
inference = 'npl_mmd'
likelihood = 'multivariate_normal'
num_posterior_samples = 21
lengthscale = -1

theta_sample = sample_npl(
    data,
    inference,
    likelihood,
    num_posterior_samples,
    seed=replication,
    lengthscale=lengthscale,
    dim=5,
    generator=generator,
)

In [ ]:
theta_sample

In [ ]:
mean=theta_sample.mean(axis=0)

In [ ]:
def cholesky_param_to_covariance(L_flat):
        """
        Converts a flattened lower triangular matrix to a covariance matrix.

        L_flat: The flattened lower triangular part of the matrix.
        dim: The dimensionality of the covariance matrix.
        """
        # Reshape the flat array into a lower triangular matrix
        L = np.zeros((5, 5))
        tril_indices = np.tril_indices(5)
        L[tril_indices] =L_flat

        # Diagonal elements of L should be strictly positive to ensure positive definiteness
        L[np.diag_indices(5)]=np.exp(jnp.diag(L))

        # Return the covariance matrix
        return L @ L.T


cholesky_param_to_covariance(mean[5:]).reshape((5,5))

In [ ]:
[[ 28.04209142 -18.89185442 -15.09456034 -23.42025597   7.92300896]
 [-18.89185442  31.68239015  19.05366462  33.48120513  -1.11980801]
 [-15.09456034  19.05366462  15.80944894  23.51607492  -2.33497905]
 [-23.42025597  33.48120513  23.51607492  60.86788088  -4.90517589]
 [  7.92300896  -1.11980801  -2.33497905  -4.90517589   4.93176023]]

In [ ]:
# from mis_dro.dataset import cont_multivariate_normal
# import scipy.spatial.distance as distance
# from mis_dro.gaussian_kernel import *
# xi = cont_multivariate_normal(80, 0.05)
# dim =5
# Xcert = np.random.uniform(np.min(xi), np.max(xi), size=[200,dim])
# zetai = np.concatenate([xi, Xcert], axis=0)
# l = np.sqrt((1/2)*np.median(distance.cdist(zetai, zetai, 'sqeuclidean')))
# K = k_jax(zetai, zetai, l)
# print(K.shape)